In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import os
import gc
import subprocess
import os
import time
from openai import OpenAI
import time
import pandas as pd
from tqdm import tqdm
import nltk
nltk.download('wordnet')
nltk.download('punkt')
import time
import pandas as pd
from tqdm import tqdm
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from bert_score import BERTScorer
import sacrebleu

In [2]:
dumps_list = [k.replace(".xlsx", "").replace("VLLM_TEST_with_metrics_adapter_", "") for k in os.listdir("TESTS_VLLM")]

In [3]:
a = [k for k in os.listdir() if "lora" in k and k not in dumps_list]
a_final = []
for k in a:
    list_dir_k = os.listdir(k)
    if 'tokenizer.json' in list_dir_k and 'adapter_config.json' in list_dir_k and 'README.md' in list_dir_k  and 'tokenizer_config.json' in list_dir_k and 'adapter_model.safetensors' in list_dir_k:
        a_final.append(k)
    #print(k, end = "\n")
    #print(list_dir_k, end = "\n")
print(len(a_final))
print(len(a))
a = a_final

0
0


In [4]:
#ruslanmv/Medical-Llama3-8B
#ruslandev/llama-3-8b-gpt-4o-ru1.0
#meta-llama/Llama-3.1-8B-Instruct

adapters_dict = {}

for k in a:
    if "meta-llama_Llama-3.1-8B-Instruct" in k:
        adapters_dict[k] = "meta-llama/Llama-3.1-8B-Instruct"
    elif "ruslanmv_Medical-Llama3-8B" in k:
        adapters_dict[k] = "ruslanmv/Medical-Llama3-8B"
    elif "ruslandev_llama-3-8b-gpt" in k:
        adapters_dict[k] = "ruslandev/llama-3-8b-gpt-4o-ru1.0"
if len(adapters_dict) == len(a):
    print("Status ok")
    
    

Status ok


In [5]:
# Инициализация BERTScore (один раз, чтобы не пересоздавать модель)
bertscorer = BERTScorer(lang="ru", rescale_with_baseline=False)
def compute_metrics(reference, hypothesis):
    """
    Вычисляет все метрики для одной пары (эталон, ответ модели).
    reference : str – эталонный ответ
    hypothesis : str – ответ модели
    возвращает словарь с метриками
    """
    # Normalize: убираем лишние пробелы, приводим к нижнему регистру? 
    # Для Exact Match лучше не менять регистр, если важно точное совпадение.
    # Для остальных метрик обычно нормализуют.
    ref = reference.strip()
    hyp = hypothesis.strip()
    
    # Exact Match (полное совпадение строк)
    exact_match = int(ref == hyp)
    
    # BLEU (используем smoothing, т.к. предложения короткие)
    smoothing = SmoothingFunction().method1
    # BLEU ожидает список токенов, токенизируем по пробелам (простая токенизация)
    ref_tokens = ref.split()
    hyp_tokens = hyp.split()
    bleu = sentence_bleu([ref_tokens], hyp_tokens, smoothing_function=smoothing)
    
    # ROUGE (возьмём ROUGE-L для баланса)
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
    scores = scorer.score(ref, hyp)
    rouge_l = scores['rougeL'].fmeasure
    
    # METEOR (требует токенизации, используем простую)
    meteor = meteor_score([ref.split()], hyp.split())
    
    # BERTScore (возвращает Precision, Recall, F1; берём F1)
    P, R, F1 = bertscorer.score([hyp], [ref])
    bertscore_f1 = F1.item()
    
    return {
        "exact_match": exact_match,
        "bleu": bleu,
        "rouge_l": rouge_l,
        "meteor": meteor,
        "bertscore_f1": bertscore_f1
    }


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
def api_ai_answer(client, prompt):
    start_time = time.perf_counter()
    response = client.chat.completions.create(
        model="LORA_THE_BEST",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1600,
        temperature=0.0
    )
    
    end_time = time.perf_counter()
    
    # Извлекаем текст ответа
    answer = response.choices[0].message.content
    response_time = end_time - start_time
    completion_tokens = response.usage.completion_tokens
    tokens_per_second = completion_tokens / response_time if response_time > 0 else 0
    return [answer, response_time, completion_tokens,tokens_per_second]

In [ ]:
def test_model_adapter(base_model_name, adapter_path, with_adapter = True):
    if with_adapter:
        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_name,
            torch_dtype=torch.bfloat16,
            device_map="auto",
            trust_remote_code=True,
        )
        
        tokenizer = AutoTokenizer.from_pretrained(base_model_name, use_fast=True)
        tokenizer.pad_token = tokenizer.eos_token
    
        model = PeftModel.from_pretrained(base_model, adapter_path)
        merged_model = model.merge_and_unload()
        output_dir = "LORA_THE_BEST"
        merged_model.save_pretrained(output_dir)
        tokenizer.save_pretrained(output_dir)
    
        del merged_model
        del model
        del base_model
        del tokenizer
        gc.collect()
        torch.cuda.empty_cache()
        #print(torch.cuda.memory_summary())
    
        log_file = open(f"DUMPS_VLLM/vllm_server_{adapter_path}.log", "w")
        command = [
            "vllm", "serve",
            "LORA_THE_BEST",
            "--host", "0.0.0.0",
            "--port", "8001",
            "--api-key", "",
            "--served-model-name", "LORA_THE_BEST",
            "--max-model-len", "8192",
            "--max-num-seqs", "4"
        ]
        
    else:

        llama3_chat_template = (
            "{% set loop_messages = messages %}"
            "{% for message in loop_messages %}"
            "{% set content = '<|start_header_id|>' + message['role'] + '<|end_header_id|>\n\n'+ message['content'] | trim + '<|eot_id|>' %}"
            "{% if loop.index0 == 0 %}"
            "{% set content = bos_token + content %}"
            "{% endif %}"
            "{{ content }}"
            "{% endfor %}"
            "{% if add_generation_prompt %}"
            "{{ '<|start_header_id|>assistant<|end_header_id|>\n\n' }}"
            "{% endif %}"
        )

        command = [
            "vllm", "serve",
            base_model_name,
            "--host", "0.0.0.0",
            "--port", "8001",
            "--api-key", "1",
            "--served-model-name", "LORA_THE_BEST",
            "--max-model-len", "8192",
            "--max-num-seqs", "4",
            "--chat-template", llama3_chat_template 
        ]
        log_file = open(f"DUMPS_VLLM/vllm_server_{base_model_name.replace('/', '_')}.log", "w")

    process = subprocess.Popen(
        command,
        stdout=log_file,
        stderr=subprocess.STDOUT,   # объединяем stderr с stdout
        text=True,                  # работаем со строками
        start_new_session=True      # отделяем процесс, чтобы он не завершился при выходе скрипта
    )
    #time.sleep(40)
    
    # Ждём строку "Application startup complete" в логе
    ready = False
    while not ready:
        time.sleep(1)
        if with_adapter:
            with open(f"DUMPS_VLLM/vllm_server_{adapter_path}.log", "r") as f:
                if "Application startup complete" in f.read():
                    ready = True
                    break
        else:
            with open(f"DUMPS_VLLM/vllm_server_{base_model_name.replace('/', '_')}.log", "r") as f:
                if "Application startup complete" in f.read():
                    ready = True
                    break
    client = OpenAI(
        base_url="http://localhost:8001/v1",
        api_key=""
    )

    #start_time = time.perf_counter()
    #response = client.chat.completions.create(
    #    model="LORA_THE_BEST",
    #    messages=[{"role": "user", "content": "Мой кариотип 46,XX,t(5;10)(q12;p22). Что это значит?"}],
    #    max_tokens=1600,
    #    temperature=0.7
    #)
    #end_time = time.perf_counter()
    #answer = response.choices[0].message.content
    #print("Ответ модели:")
    #print(answer)
    #print(f"\nВремя выполнения: {end_time - start_time:.3f} секунд")
    
    df = pd.read_excel("Q_F_T.xlsx")

    outputs = []
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Обработка запросов"):
        prompt = row['Promts']
        reference = row['References']

        answer, resp_time, comp_tokens, tps = api_ai_answer(client, prompt)
        metrics = compute_metrics(reference, answer)
        
        # Сохраняем всё вместе
        outputs.append({
            "answer": answer,
            "response_time": resp_time,
            "completion_tokens": comp_tokens,
            "tokens_per_second": tps,
            **metrics
        })

    # Распаковываем в DataFrame
    answers = [o["answer"] for o in outputs]
    response_times = [o["response_time"] for o in outputs]
    completion_tokens = [o["completion_tokens"] for o in outputs]
    tokens_per_second = [o["tokens_per_second"] for o in outputs]
    exact_match = [o["exact_match"] for o in outputs]
    bleu = [o["bleu"] for o in outputs]
    rouge_l = [o["rouge_l"] for o in outputs]
    meteor = [o["meteor"] for o in outputs]
    bertscore_f1 = [o["bertscore_f1"] for o in outputs]
    
    # Добавляем в исходный df
    df["Answers"] = answers
    df["Response times"] = response_times
    df["Completion tokens"] = completion_tokens
    df["Tokens per second"] = tokens_per_second
    df["Exact Match"] = exact_match
    df["BLEU"] = bleu
    df["ROUGE-L"] = rouge_l
    df["METEOR"] = meteor
    df["BERTScore F1"] = bertscore_f1
    if with_adapter:
    # Сохраняем результат
        df.to_excel(f"TESTS_VLLM/VLLM_TEST_with_metrics_adapter_{adapter_path}.xlsx", index=False)
    else:
        df.to_excel(f"TESTS_VLLM/VLLM_TEST_with_metrics_clear_model_{base_model_name.replace('/', '_')}.xlsx", index=False)
    
    process.terminate()
    

In [8]:
#for adapter, model in adapters_dict.items():
for adapter, model in tqdm(adapters_dict.items(), desc="Обработка адаптеров"):
    print(model, adapter)
    test_model_adapter(model, adapter)

Обработка адаптеров: 0it [00:00, ?it/s]


In [9]:
models_list_unsafe = ["ruslanmv/Medical-Llama3-8B", "ruslandev/llama-3-8b-gpt-4o-ru1.0", "meta-llama/Llama-3.1-8B-Instruct"]
used_clear = [k.replace("VLLM_TEST_with_metrics_clear_model_", "").replace(".xlsx", "").replace("_", "/") for k in os.listdir("TESTS_VLLM") if "adapter" not in k and k !=  ".ipynb_checkpoints" and ".ipynb" not in k]
models_list = [k for k in models_list_unsafe if k not in used_clear]
print(used_clear)
print(models_list)

['VLLM/TEST/RAG/ruslanmv/Medical-Llama3-8B/with/metrics', 'meta-llama/Llama-3.1-8B-Instruct', 'VLLM/TEST/RAG/ruslandev/llama-3-8b-gpt-4o-ru1.0/with/metrics', 'meta-llama/Llama-3.1-8B-Instruct/full/finetune/epochs/10/maxlen/1536/bs/3/collator/new/save', 'meta-llama/Llama-3.1-8B-Instruct/full/finetune/epochs/5/maxlen/1536/bs/3/collator/new/save', 'All/data/VLLM', 'VLLM/TEST/RAG/meta-llama/Llama-3.1-8B-Instruct/with/metrics', 'ruslandev/llama-3-8b-gpt-4o-ru1.0', 'ruslanmv/Medical-Llama3-8B']
[]


In [10]:
models_list_unsafe = ["ruslanmv/Medical-Llama3-8B", "ruslandev/llama-3-8b-gpt-4o-ru1.0", "meta-llama/Llama-3.1-8B-Instruct"]
used_clear = [k.replace("VLLM_TEST_with_metrics_clear_model_", "").replace(".xlsx", "").replace("_", "/") for k in os.listdir("TESTS_VLLM") if "adapter" not in k and k !=  ".ipynb_checkpoints" and ".ipynb" not in k]
models_list = [k for k in models_list_unsafe if k not in used_clear]
for model in tqdm(models_list, desc="Обработка моделей"):
    print(model)
    test_model_adapter(model, None, with_adapter = False)

Обработка моделей: 0it [00:00, ?it/s]


In [11]:
fft_list_unsafe = ["meta-llama_Llama-3.1-8B-Instruct_full_finetune_epochs_5_maxlen_1536_bs_3_collator_new_save",
           "meta-llama_Llama-3.1-8B-Instruct_full_finetune_epochs_10_maxlen_1536_bs_3_collator_new_save"]

used_clear = [k.replace("VLLM_TEST_with_metrics_clear_model_", "").replace(".xlsx", "") for k in os.listdir("TESTS_VLLM") if "adapter" not in k and k !=  ".ipynb_checkpoints" and ".ipynb" not in k]
fft_list = [k for k in fft_list_unsafe if k not in used_clear]


for model in tqdm(fft_list, desc="Обработка моделей"):
    print(model)
    test_model_adapter(model, None, with_adapter = False)

Обработка моделей: 0it [00:00, ?it/s]


In [ ]:
import os
import time
import subprocess
import gc
import torch
import pandas as pd
import numpy as np
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer
from tqdm import tqdm

# ---------- Конфигурация ----------
MODELS_LIST = [
    "ruslanmv/Medical-Llama3-8B",
    "ruslandev/llama-3-8b-gpt-4o-ru1.0",
    "meta-llama/Llama-3.1-8B-Instruct"
]

# Файлы
KNOWLEDGE_FILE = "данные_чистые.xlsx"
TEST_FILE = "Q_F_T.xlsx"
OUTPUT_DIR = "TESTS_VLLM_RAG"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Параметры vLLM
VLLM_HOST = "0.0.0.0"
VLLM_PORT = 8001
API_KEY = ""
MODEL_NAME = "LORA_THE_BEST"          # имя, под которым модель будет доступна через API

# Chat template для Llama 3 (общий для всех моделей, основанных на Llama 3)
LLAMA3_CHAT_TEMPLATE = (
    "{% set loop_messages = messages %}"
    "{% for message in loop_messages %}"
    "{% set content = '<|start_header_id|>' + message['role'] + '<|end_header_id|>\n\n'+ message['content'] | trim + '<|eot_id|>' %}"
    "{% if loop.index0 == 0 %}"
    "{% set content = bos_token + content %}"
    "{% endif %}"
    "{{ content }}"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "{{ '<|start_header_id|>assistant<|end_header_id|>\n\n' }}"
    "{% endif %}"
)

# ---------- Загрузка базы знаний и построение эмбеддингов ----------
print("Загрузка базы знаний...")
knowledge_df = pd.read_excel(KNOWLEDGE_FILE)
knowledge_questions = knowledge_df["Вопрос"].astype(str).tolist()
knowledge_answers = knowledge_df["Ответ"].astype(str).tolist()

print("Инициализация модели эмбеддингов...")
# Используем небольшую мультиязычную модель для CPU/GPU
embedder = SentenceTransformer('intfloat/multilingual-e5-small', device='cpu')
# Можно также 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'

print("Вычисление эмбеддингов для базы знаний...")
knowledge_embeddings = embedder.encode(knowledge_questions, show_progress_bar=True, convert_to_numpy=True)

# ---------- Функция поиска релевантного примера ----------
def retrieve_similar_question(query, top_k=1):
    """Возвращает (похожий_вопрос, ответ_на_него) для заданного вопроса."""
    query_emb = embedder.encode([query], convert_to_numpy=True)
    similarities = cosine_similarity(query_emb, knowledge_embeddings)[0]
    best_idx = np.argmax(similarities)
    return knowledge_questions[best_idx], knowledge_answers[best_idx]

# ---------- Функция формирования промпта с RAG ----------
def build_rag_prompt(user_question):
    similar_q, similar_a = retrieve_similar_question(user_question)
    prompt = (
        f"Ниже приведён пример вопроса и правильного ответа.\n\n"
        f"Вопрос: {similar_q}\n"
        f"Ответ: {similar_a}\n\n"
        f"Теперь ответь на следующий вопрос, используя аналогичный стиль и полноту.\n"
        f"Вопрос: {user_question}\n"
        f"Ответ:"
    )
    return prompt

# ---------- Функция отправки запроса к vLLM ----------
def ask_vllm(client, prompt, max_tokens=1600, temperature=0.0):
    start_time = time.perf_counter()
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=temperature
    )
    end_time = time.perf_counter()
    answer = response.choices[0].message.content
    response_time = end_time - start_time
    completion_tokens = response.usage.completion_tokens
    tokens_per_second = completion_tokens / response_time if response_time > 0 else 0
    return answer, response_time, completion_tokens, tokens_per_second

# ---------- Тестирование одной модели с RAG ----------
def test_model_with_rag(model_name, with_chat_template=True):
    print(f"\n=== Тестирование модели {model_name} с RAG ===")
    
    # Запуск vLLM сервера
    command = [
        "vllm", "serve",
        model_name,
        "--host", VLLM_HOST,
        "--port", str(VLLM_PORT),
        "--api-key", API_KEY,
        "--served-model-name", MODEL_NAME,
        "--max-model-len", "8192",
        "--max-num-seqs", "4"
    ]
    if with_chat_template:
        command.extend(["--chat-template", LLAMA3_CHAT_TEMPLATE])
    
    log_file = open(os.path.join(OUTPUT_DIR, f"vllm_server_{model_name.replace('/', '_')}.log"), "w")
    process = subprocess.Popen(command, stdout=log_file, stderr=subprocess.STDOUT, text=True, start_new_session=True)
    
    # Ожидание запуска сервера
    ready = False
    while not ready:
        time.sleep(1)
        with open(log_file.name, "r") as f:
            if "Application startup complete" in f.read():
                ready = True
                break
    
    client = OpenAI(base_url=f"http://localhost:{VLLM_PORT}/v1", api_key=API_KEY)
    
    # Загрузка тестовых данных
    test_df = pd.read_excel(TEST_FILE)
    outputs = []
    
    for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Обработка запросов"):
        user_prompt = row['Promts']
        reference = row['References']
        
        # Формируем RAG-промпт
        rag_prompt = build_rag_prompt(user_prompt)
        
        answer, resp_time, comp_tokens, tps = ask_vllm(client, rag_prompt)
        
        # Здесь можно добавить вычисление метрик, как в исходном коде,
        # но для краткости сохраняем основные поля.
        outputs.append({
            "answer": answer,
            "response_time": resp_time,
            "completion_tokens": comp_tokens,
            "tokens_per_second": tps,
            "reference": reference
        })
    
    # Сохранение результатов
    result_df = test_df.copy()
    result_df["Answers_RAG"] = [o["answer"] for o in outputs]
    result_df["Response times"] = [o["response_time"] for o in outputs]
    result_df["Completion tokens"] = [o["completion_tokens"] for o in outputs]
    result_df["Tokens per second"] = [o["tokens_per_second"] for o in outputs]
    
    out_filename = f"VLLM_TEST_RAG_{model_name.replace('/', '_')}.xlsx"
    result_df.to_excel(os.path.join(OUTPUT_DIR, out_filename), index=False)
    print(f"Результаты сохранены в {out_filename}")
    
    # Остановка сервера
    process.terminate()
    log_file.close()
    # Небольшая задержка для освобождения порта
    time.sleep(5)

# ---------- Запуск для всех моделей ----------
if __name__ == "__main__":
    # Проверяем, какие модели уже протестированы (чтобы не перезапускать лишний раз)
    tested = [f.replace("VLLM_TEST_RAG_", "").replace(".xlsx", "")
              for f in os.listdir(OUTPUT_DIR) if f.startswith("VLLM_TEST_RAG_")]
    
    for model in MODELS_LIST:
        if model in tested:
            print(f"Модель {model} уже протестирована, пропускаем.")
            continue
        # Для всех моделей используем chat template, кроме случая, если модель его не поддерживает.
        # Все три основаны на Llama 3, поэтому шаблон подходит.
        test_model_with_rag(model, with_chat_template=True)
    
    print("Все тесты завершены.")

Загрузка базы знаний...
Инициализация модели эмбеддингов...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Вычисление эмбеддингов для базы знаний...


Batches:   0%|          | 0/33 [00:00<?, ?it/s]


=== Тестирование модели ruslanmv/Medical-Llama3-8B с RAG ===


In [ ]:
# Список файлов (укажите реальные пути)
file_paths = [
    "VLLM_TEST_RAG_meta-llama_Llama-3.1-8B-Instruct.xlsx",
    "VLLM_TEST_RAG_ruslandev_llama-3-8b-gpt-4o-ru1.0.xlsx",
    "VLLM_TEST_RAG_ruslanmv_Medical-Llama3-8B.xlsx"
]

for file_path in file_paths:
    print(f"Обработка файла: {file_path}")
    df = pd.read_excel(file_path, engine='openpyxl')
    
    # Проверяем наличие нужных столбцов
    if 'References' not in df.columns or 'Answers_RAG' not in df.columns:
        print(f"  Пропущен: нет столбцов 'References' или 'Answers_RAG'")
        continue
    
    # Список для хранения результатов метрик
    metrics_list = []
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Вычисление метрик"):
        ref = row['References']
        hyp = row['Answers_RAG']
        metrics = compute_metrics(ref, hyp)
        metrics_list.append(metrics)
    
    # Разворачиваем список словарей в DataFrame и объединяем
    metrics_df = pd.DataFrame(metrics_list)
    df = pd.concat([df, metrics_df], axis=1)
    
    # Сохраняем (перезаписываем исходный файл или создаём новый)
    out_path = file_path.replace(".xlsx", "_with_metrics.xlsx")
    df.to_excel(out_path, index=False)
    print(f"  Сохранено: {out_path}")